# 08 Model Explainability (SHAP)
**Enterprise HR AI — Workforce Intelligence & Upskilling Platform**

### Purpose:
Generate global feature importance explanations and local per-employee SHAP force/waterfall values.


In [2]:
import os
import shap
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier

DATA_PROCESSED = "../data/processed"
df = pd.read_csv(os.path.join(DATA_PROCESSED, "attrition_features_engineered.csv"))

SENSITIVE_ATTRS = ['gender', 'marital_status']
TARGET_COLS = ['attrition', 'attrition_binary']
ID_COLS = ['employee_id']

feature_cols = [c for c in df.columns if c not in SENSITIVE_ATTRS + TARGET_COLS + ID_COLS]
X = df[feature_cols]
y = df['attrition_binary']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ]
)

X_train_trans = preprocessor.fit_transform(X_train)
X_test_trans = preprocessor.transform(X_test)

cat_feature_names = preprocessor.named_transformers_['cat'].get_feature_names_out(cat_cols)
all_feature_names = num_cols + list(cat_feature_names)

rf_model = RandomForestClassifier(n_estimators=200, max_depth=8, class_weight='balanced', random_state=42)
rf_model.fit(X_train_trans, y_train)

# Compute Tree SHAP values
explainer = shap.TreeExplainer(rf_model)
shap_values = explainer.shap_values(X_test_trans)


In [3]:
# Global Feature Importance
# If binary classification, class 1 shap values
if isinstance(shap_values, list):
    shap_vals_class1 = shap_values[1]
elif len(shap_values.shape) == 3:
    shap_vals_class1 = shap_values[:, :, 1]
else:
    shap_vals_class1 = shap_values

mean_abs_shap = np.abs(shap_vals_class1).mean(axis=0)
top_features_df = pd.DataFrame({
    'Feature': all_feature_names,
    'Mean_SHAP_Importance': mean_abs_shap
}).sort_values(by='Mean_SHAP_Importance', ascending=False)

print("=== TOP 10 GLOBAL ATTRITION DRIVERS ===")
print(top_features_df.head(10).to_string(index=False))


=== TOP 10 GLOBAL ATTRITION DRIVERS ===
               Feature  Mean_SHAP_Importance
                   age              0.032685
        monthly_income              0.030283
    stock_option_level              0.028522
          over_time_No              0.028493
         over_time_Yes              0.028445
composite_satisfaction              0.026811
      over_time_binary              0.025897
      years_at_company              0.023570
   total_working_years              0.020737
             job_level              0.019685


In [4]:
# Local Explanation for Employee 0 in Test Set
sample_idx = 0
sample_employee_id = df.iloc[X_test.index[sample_idx]]['employee_id']
sample_prob = rf_model.predict_proba(X_test_trans[sample_idx:sample_idx+1])[0, 1]

local_contributions = pd.DataFrame({
    'Feature': all_feature_names,
    'SHAP_Value': shap_vals_class1[sample_idx]
}).sort_values(by='SHAP_Value', key=abs, ascending=False)

print(f"Local Explanation for Employee ID {sample_employee_id} (Attrition Probability: {sample_prob:.2%}):")
print(local_contributions.head(5).to_string(index=False))


Local Explanation for Employee ID 1495 (Attrition Probability: 56.50%):
                   Feature  SHAP_Value
       total_working_years    0.078377
            monthly_income    0.061214
business_travel_Non-Travel   -0.054404
          years_at_company    0.052237
   years_with_curr_manager    0.045591
